# Notebook 3: LLM repair

This notebook repairs the damaged sentences with an open-weight language model hosted by GWDG (SAIA API).
The model is not trained here: it gets the sentence with its slots, 3 solved examples and an instruction,
and returns one word per slot. Two variants are run:

- **few-shot**: the instruction, 3 solved dev examples, and the sentence;
- **few-shot + article**: the same, plus the whole newspaper article the sentence comes from.
  This variant is only compared with the LLM's own few-shot result, never with BERT, because BERT
  sees the sentence only.

**Input.** The frozen damaged dataset `data/processed/corrupted_v2.jsonl` (hash checked against
`runs/corrupted_v2.sha256`), BLN600 in `data/raw/BLN600/` (for the articles), the prompt files in
`prompts/`, `configs/llm.yaml` (model, few-shot examples), and `SAIA_BASE_URL` and `SAIA_API_KEY` in
`.env` (never printed).

**Output.** The stored answers of both variants and of the contamination probe in `runs/preds/`. The
scoring of these answers, together with BERT, is in notebook 4.

`SPLIT` chooses the rows. It is `dev` here; the test split stays closed until `ALLOW_TEST` is set and
`runs/repair_config_v1.yaml` exists. A stored answer is never asked again, so rerunning the notebook
sends no new request for rows that are already stored.

In [1]:
import hashlib
import json
import re
import time

import pandas as pd
import pysbd
from rapidfuzz.distance import Levenshtein

from blnrepair.data import ROOT, load_config, load_gold, split_sentences
from blnrepair.freeze import load_frozen
from blnrepair.llm import SaiaClient, variant_versions
from blnrepair.preds import append_pred, load_preds, pred_path, row_key, run_method
from blnrepair.slots import build_slots, slot_core, slot_view

pd.set_option("display.max_colwidth", None)
config = load_config()
rows = load_frozen("v2")  # stops if the file hash does not match runs/corrupted_v2.sha256
by_key = {(r["id"], r["severity"]): r for r in rows}

SPLIT, ALLOW_TEST = "dev", False
if SPLIT == "test":
    assert ALLOW_TEST and (ROOT / "runs/repair_config_v1.yaml").exists(), "the test split is closed"

## 1. The model

The connection to SAIA is in `src/blnrepair/llm.py` (plumbing, not method). It allows only the model ids
listed in `configs/llm.yaml`, sends every request at temperature 0 with at most 1,024 answer tokens, and
sends a request again only when the server is busy or over its rate limit, never because of what an
answer says. The model is Llama 3.1 8B Instruct, open-weight and run by GWDG.

In [2]:
client = SaiaClient()
MODEL = client.cfg["model"]
client.check_model(MODEL)
print(MODEL)

meta-llama-3.1-8b-instruct


## 2. The prompt and the answer

**The prompt** is one prompt file per variant. Its parts: a short system message (an editor who answers
only in the requested format and writes no code); the instruction (the text is 19th-century British
newspaper English and spelling is not to be modernised; how a damaged and a missing word look; every slot
needs a best guess; the answer is only a JSON list with one word per slot); **3 solved examples**; for
the third variant the **article**; and the sentence in the same slot view that BERT gets. A prompt file is
never edited once used. The system message was added in version 2: without it, this model answered the
version 1 prompt with Python code in all 15 trial requests.

The 3 examples are dev sentences at levels 1w, 25 and 75, chosen to show a name, a dropped word, a dropped
comma, a sum of money and punctuation that is added back. The answer of an example is each gold word
without its edge punctuation, the form the model is asked to return. Their gold answers are in the prompt,
so **these 3 sentences are never scored**. On the test split this leaves nothing out: the examples are dev
sentences, and dev and test share no excerpt.

**The answer** is read with only this cleanup: strip whitespace and markdown code fences, then take the
first JSON array in the text. It is a **format failure** if there is no array, if it has the wrong number
of items, if an item is not a string, or if the model was cut off. A format failure is stored as it is and
scored as no repair in notebook 4; there is no retry and no manual fix. An item with a space inside is kept
and counted as `multiword`.

In [3]:
TEMPLATES = {name: (ROOT / "prompts" / f"{file}.txt").read_text(encoding="utf-8")
             for name, file in [("few-shot", "repair_fewshot_v2"), ("few-shot + article", "repair_fewshot_context_v1")]}
FEWSHOT_PICKS = client.cfg["fewshot_picks"]


def example_text(row):
    answer = [slot_core(p["gold"]) for p in sorted(row["ops_per_word"], key=lambda p: p["idx"])]
    return f"Sentence:\n{slot_view(build_slots(row))}\nAnswer: {json.dumps(answer, ensure_ascii=False)}"


EXAMPLES = "\n\n".join(example_text(by_key[tuple(pick)]) for pick in FEWSHOT_PICKS)
example_ids = {i for i, _ in FEWSHOT_PICKS}


def build_messages(template, view, examples="", context=""):
    # A prompt file holds a [system] part and a [user] part; the user part gets the placeholders filled in.
    system, user = template.removeprefix("[system]\n").split("\n[user]\n")
    values = {"examples": examples, "context": context, "k": str(view["k"]), "sentence": slot_view(view)}
    user = re.sub(r"\{(examples|context|k|sentence)\}", lambda m: values[m.group(1)], user)
    return [{"role": "system", "content": system.strip()}, {"role": "user", "content": user}]


def parse_answer(content, finish_reason, k):
    # (list or None, error text or None); a wrong-length list is returned too, and the store marks it as a failure
    if finish_reason == "length":
        return None, "answer cut off (finish_reason is length)"
    text = "\n".join(line for line in content.strip().splitlines() if not line.strip().startswith("```"))
    decoder = json.JSONDecoder()
    for start in (m.start() for m in re.finditer(r"\[", text)):
        try:
            words, _ = decoder.raw_decode(text[start:])
        except json.JSONDecodeError:
            continue
        if len(words) != k:
            return words, f"expected {k} items, got {len(words)}"
        if not all(isinstance(w, str) for w in words):
            return words, "an item is not a string"
        return words, None
    return None, "no JSON array in the answer"


show = by_key[("3200812175-012", "25")]
for message in build_messages(TEMPLATES["few-shot"], build_slots(show), EXAMPLES):
    print(f"[{message['role']}]\n{message['content']}\n")

[system]
You are a careful editor of 19th-century British newspaper text. You answer only in the format the user asks for, and you never write code.

[user]
The sentence below comes from a 19th-century British newspaper. It was scanned and read by OCR, and some of its words are damaged. Each damaged word is shown between ⟨ and ⟩ exactly as the scan gave it, with letters changed or missing. A word that is missing completely is shown as ⟨?⟩. All other words are correct, and so is the punctuation next to a damaged word.

The text is Victorian British English. Do not modernize the spelling, and write names the way a newspaper of that time would.

For each damaged word, give the word that was originally printed there, in the order in which they appear. Always give your best guess, even when you are unsure. The punctuation next to a damaged word is added back automatically, so give only the word. For ⟨?⟩ give the missing word, or the punctuation mark if that is what is missing. Return only a

## 3. The article as context

For the third variant the prompt also holds the whole article (BLN600 excerpt) the sentence comes from. The
article is cut into sentences with the same splitter as in notebook 1; the sentence to repair is replaced
by its slot view and marked with `<<< >>>`, so the model never sees its gold words. The other sentences are
the gold text. Train, dev and test are split by excerpt, so a dev article holds no test sentence.

In [4]:
gold = load_gold(ROOT / config["paths"]["raw_dir"])
segmenter = pysbd.Segmenter(language="en", clean=False)
assert not {r["doc_id"] for r in rows if r["split"] == "dev"} & {r["doc_id"] for r in rows if r["split"] == "test"}


def article_context(row):
    target, i, paragraphs = int(row["id"].rsplit("-", 1)[1]), 0, []
    for paragraph in gold[row["doc_id"]]:
        sentences = []
        for sentence in split_sentences(paragraph, segmenter):
            if i == target:
                assert sentence == row["gold_text"], row["id"]
                sentence = f"<<< {slot_view(build_slots(row))} >>>"
            sentences.append(sentence)
            i += 1
        paragraphs.append(" ".join(sentences))
    return "\n\n".join(paragraphs)


scored = [r for r in rows if r["split"] == SPLIT and r["id"] not in example_ids]
scored_ids = sorted({r["id"] for r in scored})
words = pd.Series({i: len(article_context(by_key[(i, "25")]).split()) for i in scored_ids})
print(f"{len(scored_ids)} sentences of the {SPLIT} split are repaired (the 3 examples are left out)")
print(f"article length in words: min {words.min()}, median {words.median():.0f}, max {words.max()}")
message = build_messages(TEMPLATES["few-shot + article"], build_slots(show), EXAMPLES, article_context(show))[1]
print("\n... end of the article variant's prompt:\n" + message["content"][-1100:])

17 sentences of the dev split are repaired (the 3 examples are left out)
article length in words: min 197, median 591, max 892

... end of the article variant's prompt:
ell in the world, and the prisoner’s behaviour had been a terrible upset to them all. He was quite willing to send him to see if the magistrate could see his way to admonish him, and then allow him to take him away. Mr. Cooke remarked that the prisoner was a very bad boy. He could not permit that course to be taken. The boy had much better go to a reformatory. Detective Collins said that the prisoner had stayed at three hotels, and had absconded without paying his bills. They, however, had now been paid. Mr. Cooke sentenced the prisoner to ten days’ imprisonment, to be followed by four years’ detention on the reformatory ship Cornwall, the magistrate observing that it was necessary the prisoner should go to prison before the authorities would receive him on the ship.

Sentence:
It was ascertained that he had hired a bic

## 4. Running the variants

`make_predict` turns a prompt into the function the prediction store calls for each row. The store is the
same as for BERT: every answer is saved in `runs/preds/`, and a stored row is never asked again. The raw
answer, `finish_reason`, the model id the API reports, the token counts and the `multiword` positions are
kept. Both variants run on every row of the split with slots, at all five damage levels (level 0 has
nothing to repair). When the rate limit is reached, the client waits until the server's reset time; the
`seconds` of such a row include the wait.

In [5]:
def make_predict(template, examples="", with_article=False):
    def predict(view):
        context = article_context(by_key[(view["id"], view["severity"])]) if with_article else ""
        out = client.chat(MODEL, build_messages(template, view, examples, context))
        words, error = parse_answer(out["content"], out["finish_reason"], view["k"])
        multiword = [i for i, w in enumerate(words or []) if isinstance(w, str) and " " in w.strip()]
        raw = {k: out[k] for k in ("content", "finish_reason", "model", "prompt_tokens", "completion_tokens")}
        return {"raw_output": json.dumps({**raw, "multiword": multiword}, ensure_ascii=False),
                "pred_words": words, "error": error}
    return predict


versions = variant_versions(client.cfg, SPLIT)
for name in ("few-shot", "few-shot + article"):
    method, version = versions[name]
    added = run_method(scored, make_predict(TEMPLATES[name], EXAMPLES, name == "few-shot + article"),
                       method, version, MODEL, allow_test=ALLOW_TEST)
    print(f"{name}: {len(load_preds(pred_path(method, version)))} rows stored ({len(added)} new requests)")
print(f"requests sent in this run, retries included: {client.calls}")

few-shot: 85 rows stored (0 new requests)
few-shot + article: 85 rows stored (0 new requests)
requests sent in this run, retries included: 0


## 5. Contamination probe

BLN600 is public, so the model may have seen these articles in training. The probe gives the model the
first half of each clean sentence of the split (by words) and asks it to continue. The continuation is cut
to the length of the real second half; a letter similarity of 0.9 or more to the real second half counts as
near-verbatim.

In [6]:
PROBE = ("Here is the beginning of a sentence from a 19th-century British newspaper. Continue it. "
         "Write only the rest of the sentence, nothing else.\n\n{start}")


def run_probe(clean_rows, version):
    path = pred_path("llm_probe", version)
    done = {row_key(p) for p in load_preds(path)}
    for row in clean_rows:
        assert row["split"] == SPLIT and (SPLIT != "test" or ALLOW_TEST)
        if (row["id"], row["severity"], "llm_probe", version) in done:
            continue
        words = row["gold_tokens"]
        half = len(words) // 2
        start, gold_rest = " ".join(words[:half]), " ".join(words[half:])
        t0 = time.perf_counter()
        out = client.chat(MODEL, [{"role": "user", "content": PROBE.replace("{start}", start)}])
        continuation = " ".join(out["content"].split()[:len(words) - half])
        similarity = Levenshtein.normalized_similarity(continuation, gold_rest)
        raw = {"content": out["content"], "finish_reason": out["finish_reason"], "model": out["model"],
               "similarity": round(similarity, 3), "near_verbatim": similarity >= 0.9}
        append_pred(path, {"id": row["id"], "severity": row["severity"], "split": row["split"],
                           "method": "llm_probe", "method_version": version, "model_name": MODEL,
                           "slots_in": start, "raw_output": json.dumps(raw, ensure_ascii=False),
                           "pred_words": [continuation], "spliced_text": continuation, "format_ok": True,
                           "error": None, "seconds": round(time.perf_counter() - t0, 3),
                           "timestamp": pd.Timestamp.now(tz="UTC").isoformat(timespec="seconds")})


probe_version = versions["probe"][1]
run_probe([r for r in rows if r["split"] == SPLIT and r["severity"] == "0"], probe_version)
probe = pd.DataFrame([json.loads(p["raw_output"]) for p in load_preds(pred_path("llm_probe", probe_version))])
print(f"probe on {len(probe)} {SPLIT} sentences: {probe['near_verbatim'].sum()} near-verbatim; "
      f"similarity median {probe['similarity'].median():.2f}, max {probe['similarity'].max():.2f}")

probe on 20 dev sentences: 0 near-verbatim; similarity median 0.24, max 0.60
